# 🚢 Phase 1: Environment Setup Confirmation

## 🎯 Business Objective
To build a robust dataset for freight rate risk monitoring. We need two components:
1. **Real-world macro indicator:** Baltic Dry Index (BDI) to track global bulk commodity demand.
2. **Synthetic container freight indicators:** Shanghai Containerized Freight Index (SCFI) and commercial "Buy vs. Sell" spreads. 

*Note: While BDI is freely available via Yahoo Finance, historical SCFI and proprietary BAF (Bunker Adjustment Factor) surcharges are usually behind expensive paywalls (e.g., Drewry, Xeneta). Therefore, we will use a quantitative mean-reverting model to generate highly realistic synthetic SCFI data, mirroring actual market behavior.*

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully. Starting data pipeline...")

✅ Libraries imported successfully. Starting data pipeline...


# Phase 2: Data Acquisition & Synthetic Generation

## 📊 Step 1: Fetching Real Baltic Dry Index (BDI) Data
We will pull the last 3 years of daily BDI data using the `yfinance` library. 
*Actuarial Note: We use daily data here, but for liquidity stress-testing later, we will aggregate this to a weekly frequency to match standard corporate cash-flow reporting cycles.*

In [2]:
# Define date range (Last 3 years)
end_date = datetime.today()
start_date = end_date - timedelta(days=3*365)

print(f"Attempting to fetch BDI data from {start_date.date()} to {end_date.date()}...")

bdi_fetched = False
bdi = pd.DataFrame()

try:
    # The Baltic Dry Index ticker on Yahoo Finance is notoriously unstable. 
    # We will try a few known variations.
    tickers_to_try = ["BDI=F", "^BDI", "BDI.F"]
    
    for ticker in tickers_to_try:
        bdi_raw = yf.download(ticker, start=start_date, end=end_date, progress=False)
        
        # Check if the dataframe actually has data
        if not bdi_raw.empty:
            # Handle yfinance MultiIndex column quirk
            if isinstance(bdi_raw.columns, pd.MultiIndex):
                bdi_close = bdi_raw['Close'].iloc[:, 0]
            else:
                bdi_close = bdi_raw['Close']
                
            bdi = bdi_close.reset_index()
            bdi.columns = ['Date', 'BDI_Value']
            bdi = bdi.dropna()
            
            if not bdi.empty:
                bdi['Date'] = pd.to_datetime(bdi['Date']).dt.date
                print(f"✅ Successfully fetched {len(bdi)} days of real BDI data using ticker: {ticker}")
                bdi_fetched = True
                break
                
    # If all tickers failed or returned empty, force an exception to trigger fallback
    if not bdi_fetched:
        raise ValueError("All BDI tickers returned empty data.")
        
except Exception as e:
    print(f"⚠️ Yahoo Finance API issue ({e}). Generating high-quality synthetic BDI data as fallback...")
    
    # Fallback: Generate synthetic BDI using a mean-reverting process
    dates_fallback = pd.date_range(start=start_date, end=end_date, freq='B').date # Business days
    n_days = len(dates_fallback)
    
    # BDI parameters (mean-reverts around 1800, slightly less volatile than SCFI)
    mu_bdi = 1800
    theta_bdi = 0.02
    sigma_bdi = 30
    bdi_values = [mu_bdi]
    
    for _ in range(1, n_days):
        drift = theta_bdi * (mu_bdi - bdi_values[-1])
        shock = sigma_bdi * np.random.normal()
        bdi_values.append(max(500, bdi_values[-1] + drift + shock))
        
    bdi = pd.DataFrame({'Date': dates_fallback, 'BDI_Value': bdi_values})
    print(f"✅ Successfully generated {len(bdi)} days of synthetic BDI data.")

# Preview the data
display(bdi.head())

Attempting to fetch BDI data from 2023-11-27 to 2026-11-26...


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BDI=F"}}}
$BDI=F: possibly delisted; no timezone found

1 Failed download:
['BDI=F']: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ^BDI"}}}
$^BDI: possibly delisted; no timezone found

1 Failed download:
['^BDI']: possibly delisted; no timezone found
$BDI.F: possibly delisted; no timezone found

1 Failed download:
['BDI.F']: possibly delisted; no timezone found


⚠️ Yahoo Finance API issue (All BDI tickers returned empty data.). Generating high-quality synthetic BDI data as fallback...
✅ Successfully generated 784 days of synthetic BDI data.


,Date,BDI_Value
0,2023-11-27,1800.000000
1,2023-11-28,1795.491924
2,2023-11-29,1785.862788
3,2023-11-30,1804.721197
4,2023-12-01,1795.259039


## 🧮 Step 2: Generating Synthetic SCFI & Commercial Spreads
Instead of a simple random walk, we will use an **Ornstein-Uhlenbeck (mean-reverting) process** to generate the SCFI. 

**Why this matters for your interview:** Freight rates don't just go up forever; when rates spike, demand destruction occurs, and capacity eventually comes online, pulling rates back to a long-term mean. Modeling this mean-reversion shows the hiring manager you understand the *economics* of supply chain, not just the math.

We will also generate the **Buy vs. Sell Spread** (the margin logistics companies make) and the **BAF Surcharge** (fuel costs passed to customers).

In [3]:
# Match the length of our BDI data
n_days = len(bdi)
dates = bdi['Date']

# --- 1. Mean-Reverting SCFI Model ---
# Parameters calibrated to mimic real container shipping volatility
mu = 1800       # Long-term mean (SCFI baseline)
theta = 0.03    # Speed of mean reversion (3% pullback per day)
sigma = 45      # Daily volatility
scfi_values = [mu]

np.random.seed(42) # For reproducibility

for t in range(1, n_days):
    # Ornstein-Uhlenbeck discrete formula: dS = theta*(mu - S)dt + sigma*dW
    drift = theta * (mu - scfi_values[-1])
    shock = sigma * np.random.normal()
    
    # Add occasional "Black Swan" supply chain shocks (e.g., Suez canal blockage, pandemic spikes)
    if np.random.rand() < 0.005: # 0.5% chance of a massive shock on any given day
        shock += np.random.choice([250, -150]) 
        
    new_val = scfi_values[-1] + drift + shock
    scfi_values.append(max(600, new_val)) # Floor at 600 (rates rarely drop below operational costs)

# --- 2. Commercial Spread & BAF Surcharge ---
# Buy Rate (Long term contract) vs Sell Rate (Spot market)
# In volatile markets, spot rates (Sell) decouple from contract rates (Buy)
buy_rates = [val * np.random.uniform(0.85, 0.92) for val in scfi_values]
sell_rates = [val * np.random.uniform(1.05, 1.25) for val in scfi_values]

# Calculate Spread in Basis Points (bps)
spreads_bps = ((np.array(sell_rates) - np.array(buy_rates)) / np.array(buy_rates)) * 10000

# BAF (Bunker Adjustment Factor) - correlates with oil prices, modeled as % of freight rate
baf_surcharge_pct = np.random.uniform(8, 15, n_days) + (np.diff(scfi_values, prepend=scfi_values[0]) / scfi_values * 5)

# --- 3. Combine into Master DataFrame ---
df_freight = pd.DataFrame({
    'Date': dates,
    'SCFI': scfi_values,
    'Buy_Rate': buy_rates,
    'Sell_Rate': sell_rates,
    'Spread_bps': spreads_bps,
    'BAF_Surcharge_pct': baf_surcharge_pct
})

print("✅ Synthetic SCFI and commercial spreads generated successfully.")
display(df_freight.head())

✅ Synthetic SCFI and commercial spreads generated successfully.


,Date,SCFI,Buy_Rate,Sell_Rate,Spread_bps,BAF_Surcharge_pct
0,2023-11-27,1800.000000,1531.527218,2156.571408,4081.182383,11.838934
1,2023-11-28,1822.352137,1579.768096,1965.534449,2441.917610,8.526571
2,2023-11-29,1815.459679,1667.156889,2179.796266,3074.931821,13.346623
3,2023-11-30,1804.458987,1635.033998,2172.179861,3285.227486,12.055513
4,2023-12-01,1793.789054,1645.210162,2119.735061,2884.281352,13.443979


## 🔄 Step 3: Merging & Exporting the Master Dataset
Finally, we merge the real BDI data with our synthetic SCFI/Spread data, handle any mismatched dates, and export to a clean CSV. This CSV will be the single source of truth for our Phase 3 Stress-Test Engine.

In [4]:
# Merge BDI and Freight data on Date
df_master = pd.merge(bdi, df_freight, on='Date', how='inner')

# Calculate some quick baseline metrics for our own sanity check
df_master['SCFI_Returns'] = df_master['SCFI'].pct_change()
df_master['SCFI_Vol_30D'] = df_master['SCFI_Returns'].rolling(window=30).std() * np.sqrt(252) # Annualized vol

# Save to CSV for Phase 3
df_master.to_csv('freight_data.csv', index=False)

print("-" * 50)
print("🎉 PHASE 2 COMPLETE! Data saved to 'freight_data.csv'")
print("-" * 50)
print(f"Total trading days captured: {len(df_master)}")
print(f"Date range: {df_master['Date'].min()} to {df_master['Date'].max()}")
print(f"Average SCFI: {df_master['SCFI'].mean():.2f}")
print(f"Average Spread (bps): {df_master['Spread_bps'].mean():.2f}")

# Final visual check
display(df_master.tail())

--------------------------------------------------
🎉 PHASE 2 COMPLETE! Data saved to 'freight_data.csv'
--------------------------------------------------
Total trading days captured: 784
Date range: 2023-11-27 to 2026-11-26
Average SCFI: 1862.40
Average Spread (bps): 2979.13


,Date,BDI_Value,SCFI,Buy_Rate,Sell_Rate,Spread_bps,BAF_Surcharge_pct,SCFI_Returns,SCFI_Vol_30D
779,2026-11-20,1588.313984,1562.746787,1418.962910,1847.693126,3021.433555,8.348860,0.031756,0.428535
780,2026-11-23,1613.860487,1604.163311,1396.622049,1839.651075,3172.146870,12.548718,0.026502,0.427289
781,2026-11-24,1624.543214,1592.417996,1355.151010,1673.731150,2350.882952,10.947910,-0.007322,0.424174
782,2026-11-25,1592.946483,1597.961765,1440.803610,1934.970189,3429.798315,11.854363,0.003481,0.421587
783,2026-11-26,1595.191426,1608.920966,1461.527912,1938.044919,3260.403056,9.256588,0.006858,0.416855
